# COVID-QU CBM Evaluation & Concept Inspection
Compare standard, LF-CBM, and VLG-CBM on COVID-QU (3-class), inspect concepts, and optionally probe CheXagent.

## 1) Configure paths
Set dataset root (folder-based), variant, and run directories.

In [ ]:
# Paths
DATA_DIR = "/workspace/datasets/COVIDQU"
VARIANT = "lung"  # or "infection"
STANDARD_CKPT = "checkpoints/covidqu_backbone/best_model.pth"  # from train.py
LF_RUN_DIR = "checkpoints/lfcbm_covidqu_lung"  # LF-CBM artifacts
VLG_RUN_DIR = "checkpoints/vlg_cbm_covidqu_lung"  # VLG-CBM artifacts + annotations
OUTPUT_DIR = "notebook_eval/covidqu"
IMG_SIZE = 224
BATCH_SIZE = 256
NUM_WORKERS = 12


## 2) Standard model eval (accuracy)
Use eval.py with `--label_set covidqu`.

In [ ]:
import os, json, subprocess
os.makedirs(OUTPUT_DIR, exist_ok=True)

split = "test"  # Train/Val/Test folder mapping handled by eval.py folder loader
cmd = [
    "python", "eval.py",
    "--data_dir", DATA_DIR,
    "--label_set", "covidqu",
    "--covidqu_variant", VARIANT,
    "--model", "densenet121",
    "--checkpoint", STANDARD_CKPT,
    "--output", os.path.join(OUTPUT_DIR, f"standard_{split}"),
    "--img_size", str(IMG_SIZE),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--split", split,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
metrics_path = os.path.join(OUTPUT_DIR, f"standard_{split}", f"{split}_metrics.json")
with open(metrics_path) as f:
    std_metrics = json.load(f)
print("Standard model accuracy:", std_metrics.get("accuracy", "N/A"))


## 3) LF-CBM evaluation (accuracy)
Uses evaluate_nec.py full-model accuracy.

In [ ]:
split = "test"
cmd = [
    "python", "evaluate_nec.py",
    "--model_dir", LF_RUN_DIR,
    "--data_dir", DATA_DIR,
    "--label_set", "covidqu",
    "--covidqu_variant", VARIANT,
    "--split", split,
    "--batch_size", str(BATCH_SIZE),
    "--nec_levels", "60",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
# Accuracy printed by script and stored in nec_metrics.csv


## 4) VLG-CBM evaluation (accuracy)
Requires annotation dirs configured in VLG_RUN_DIR config/arguments.

In [ ]:
split = "test"
cmd = [
    "python", "evaluate_nec.py",
    "--model_dir", VLG_RUN_DIR,
    "--data_dir", DATA_DIR,
    "--label_set", "covidqu",
    "--covidqu_variant", VARIANT,
    "--split", split,
    "--batch_size", str(BATCH_SIZE),
    "--nec_levels", "60",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 5) Concept contributions (LF/VLG)
Load W_g and concepts to view top positive/negative concepts per class.

In [ ]:
import torch, numpy as np, os
from dataset import COVIDQU_LABELS

def load_concepts(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

def top_concepts(run_dir, labels, topk=5):
    W_g = torch.load(os.path.join(run_dir, "W_g.pt"))
    concepts = load_concepts(os.path.join(run_dir, "concepts.txt"))
    W_np = W_g.cpu().numpy()
    for cls_idx, cls in enumerate(labels):
        weights = W_np[cls_idx]
        pos_idx = np.argsort(weights)[-topk:][::-1]
        neg_idx = np.argsort(weights)[:topk]
        print(f"\n{cls} (run={run_dir}):")
        print("  Positive:")
        for i in pos_idx:
            print(f"    + {concepts[i]}: {weights[i]:.3f}")
        print("  Negative:")
        for i in neg_idx:
            print(f"    - {concepts[i]}: {weights[i]:.3f}")

top_concepts(LF_RUN_DIR, COVIDQU_LABELS)


## 6) CheXagent sanity check for top concepts (optional)
Loads CheXagent HF model and returns vision embedding; plug in text encoder (e.g., Mistral embeddings) for similarity checks.

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

CHEX_MODEL = "StanfordAIMI/CheXagent-8b"
chex_processor = AutoProcessor.from_pretrained(CHEX_MODEL, trust_remote_code=True)
chex_model = AutoModelForCausalLM.from_pretrained(CHEX_MODEL, trust_remote_code=True, torch_dtype=torch.float16).to("cuda").eval()

sample_image = "/path/to/covidqu/image.png"
img = Image.open(sample_image).convert("RGB")
inputs = chex_processor(images=[img], return_tensors="pt").to("cuda")
with torch.no_grad():
    vision_out = chex_model.model.vision_model(pixel_values=inputs["pixel_values"])
    vision_feat = vision_out.last_hidden_state.mean(dim=1)
    vision_feat = torch.nn.functional.normalize(vision_feat, dim=1)

print("CheXagent vision embedding:", vision_feat.shape)
# TODO: add text embeddings for top concepts and cosine similarity.
